# EGregoRA — Colab GPU worker

Open this in Google Colab under any account, set the runtime to a **T4 GPU**
(Runtime → Change runtime type), then **Runtime → Run all**.

The notebook starts a small server on the Colab machine, opens a Cloudflare
quick tunnel to it, and registers that address with the site. From then on the
oracle sends clips here to be rendered. Nothing about your Google account is
shared — the site only ever learns a temporary tunnel URL.

Close the tab or stop the last cell and the worker drops out of the pool.

**What this worker now reports back.** While a clip renders it sends its own
step count to the site, so the bar on the Ask Ed page shows a *measured*
percentage rather than a guess, and the name of the card it is running on
(`Tesla T4`, `L4`, whatever Colab handed you) is shown beside it.

**Continuity.** When a shot is continued from an earlier frame the site sends
the seed and, where the pipeline accepts one, the starting frame itself. The
notebook uses both, so the same characters, the same place and the same camera
carry across the cut.


In [ ]:
#@title EGregoRA · GPU worker { display-mode: "form" }
#@markdown Run this cell. It wakes a GPU, opens a private tunnel, and tells
#@markdown egregora.hatchable.site that this machine is available. Leave the
#@markdown tab open; close it and the worker simply disappears from the pool.

SITE   = "https://egregora.hatchable.site"  #@param {type:"string"}
LABEL  = "Colab"                            #@param {type:"string"}
SECRET = ""                                 #@param {type:"string"}

import subprocess, sys, os, threading, time, json, uuid, re, shutil, urllib.request

assert shutil.which("nvidia-smi"), "No GPU. Runtime - Change runtime type - T4 GPU, then run again."
print(subprocess.run(["nvidia-smi","--query-gpu=name","--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip())

subprocess.run([sys.executable,"-m","pip","install","-q",
                "diffusers>=0.32.0","transformers","accelerate","sentencepiece",
                "imageio","imageio-ffmpeg","flask"], check=True)

In [ ]:
#@title Start the worker
import torch, diffusers, imageio, flask, queue, urllib.request, io
from diffusers.utils import export_to_video
from flask import Flask, request, jsonify, send_file
try:
    from PIL import Image
except Exception:
    Image = None

GPU = subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip().replace("\n", " ") or "GPU"
print("this worker is", GPU)
JOBS, WORK, CACHE = {}, queue.Queue(), {}

def load(repo, cls):
    if repo in CACHE: return CACHE[repo]
    print("loading", repo, flush=True)
    Pipe = getattr(diffusers, cls, None) or diffusers.DiffusionPipeline
    try:
        pipe = Pipe.from_pretrained(repo, torch_dtype=torch.bfloat16)
    except Exception as e:
        print("bf16 failed, trying fp16:", e, flush=True)
        pipe = Pipe.from_pretrained(repo, torch_dtype=torch.float16)
    pipe.enable_model_cpu_offload()
    for fn in ("enable_vae_tiling", "enable_vae_slicing"):
        try: getattr(pipe.vae, fn)()
        except Exception: pass
    CACHE.clear(); CACHE[repo] = pipe      # one model resident at a time
    return pipe

def fetch_image(url):
    """The first frame of a continuation, handed over by the site."""
    if not url or Image is None: return None
    try:
        with urllib.request.urlopen(url, timeout=60) as r:
            return Image.open(io.BytesIO(r.read())).convert("RGB")
    except Exception as e:
        print("could not fetch the starting frame:", e, flush=True)
        return None

def run(job):
    m = job["model"]
    pipe = load(m["repo"], m.get("cls", "DiffusionPipeline"))
    steps = int(m.get("steps", 28))

    # --- progress, measured rather than guessed -------------------------
    # The site shows a percentage bar. It is only worth showing if it is
    # real, so the sampler reports its own step count on every callback and
    # the last 8% is left for the VAE decode and the mp4 write, which are
    # not free.
    def on_step(pipe_, step, timestep, cb_kwargs):
        job["progress"] = int(4 + 88 * (step + 1) / max(1, steps))
        job["stage"] = "sampling, step %d of %d" % (step + 1, steps)
        return cb_kwargs

    kw = dict(prompt=job["prompt"],
              num_inference_steps=steps,
              num_frames=int(job.get("frames") or m.get("frames", 81)),
              callback_on_step_end=on_step)

    # --- continuity -----------------------------------------------------
    # The same seed means the same noise, which means the same world even
    # when the route cannot take a starting frame. Where the pipeline does
    # accept one, the frame itself is the strongest lock there is.
    seed = job.get("seed")
    if seed:
        kw["generator"] = torch.Generator(device="cuda").manual_seed(int(seed) % (2**31 - 1))
    init = fetch_image(job.get("init_image"))
    if init is not None:
        kw["image"] = init
        job["stage"] = "continuing from the given frame"

    if job.get("aspect") == "9:16":
        kw["width"], kw["height"] = int(m.get("h", 480)), int(m.get("w", 848))
    else:
        kw["width"], kw["height"] = int(m.get("w", 848)), int(m.get("h", 480))

    def call(**kwargs):
        try:
            return pipe(**kwargs)
        except TypeError as e:
            # older pipelines take neither a step callback nor an init image
            for drop in ("callback_on_step_end", "image", "width", "height"):
                if drop in kwargs and drop in str(e):
                    kwargs.pop(drop)
                    return call(**kwargs)
            raise

    try:
        out = call(**kw)
    except TypeError:
        for drop in ("callback_on_step_end", "image", "width", "height"):
            kw.pop(drop, None)
        out = pipe(**kw)

    job["progress"] = 94
    job["stage"] = "writing the file"
    frames = out.frames[0]
    path = f"/content/{job['id']}.mp4"
    export_to_video(frames, path, fps=int(m.get("fps", 24)))
    job["progress"] = 99
    return path

def pump():
    while True:
        jid = WORK.get()
        job = JOBS[jid]
        job["status"] = "running"
        job["progress"] = 2
        job["stage"] = "loading the model"
        try:
            job["path"] = run(job); job["status"] = "ready"; job["progress"] = 100
        except Exception as e:
            job["status"] = "failed"; job["error"] = str(e)[:400]
            print("job failed:", e, flush=True)
        finally:
            torch.cuda.empty_cache()
threading.Thread(target=pump, daemon=True).start()

app = Flask(__name__)

@app.get("/health")
def health(): return jsonify(ok=True, gpu=GPU, busy=not WORK.empty())

@app.post("/submit")
def submit():
    if SECRET and request.headers.get("x-egregora-secret") != SECRET:
        return jsonify(error="forbidden"), 403
    b = request.get_json(force=True, silent=True) or {}
    if not b.get("prompt") or not b.get("model"): return jsonify(error="prompt and model required"), 400
    jid = uuid.uuid4().hex[:12]
    JOBS[jid] = {"id": jid, "status": "queued", "progress": 1, "stage": "queued", **b}
    WORK.put(jid)
    return jsonify(id=jid)

@app.get("/job")
def job():
    j = JOBS.get(request.args.get("id", ""))
    if not j: return jsonify(status="failed", error="no such job"), 404
    return jsonify(status=j["status"], error=j.get("error"),
                   progress=j.get("progress", 0), stage=j.get("stage"), gpu=GPU)

@app.get("/file")
def file():
    j = JOBS.get(request.args.get("id", ""))
    if not j or j.get("status") != "ready": return jsonify(error="not ready"), 409
    return send_file(j["path"], mimetype="video/mp4")

threading.Thread(target=lambda: app.run(host="0.0.0.0", port=8711, threaded=True),
                 daemon=True).start()
time.sleep(2)
print("worker up on :8711")


In [ ]:
#@title Open the tunnel and join the pool
if not os.path.exists("/content/cloudflared"):
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "/content/cloudflared")
    os.chmod("/content/cloudflared", 0o755)

log = open("/content/tunnel.log", "w")
proc = subprocess.Popen(["/content/cloudflared", "tunnel", "--no-autoupdate",
                         "--url", "http://127.0.0.1:8711"],
                        stdout=log, stderr=subprocess.STDOUT)

PUBLIC = None
for _ in range(60):
    time.sleep(2)
    try: txt = open("/content/tunnel.log").read()
    except FileNotFoundError: txt = ""
    m = re.search(r"https://[-\w]+\.trycloudflare\.com", txt)
    if m: PUBLIC = m.group(0); break
assert PUBLIC, "the tunnel did not come up — run this cell again"
print("public endpoint:", PUBLIC)

def post(payload):
    req = urllib.request.Request(SITE + "/api/colab",
        data=json.dumps(payload).encode(),
        headers={"content-type": "application/json", "x-egregora-secret": SECRET})
    with urllib.request.urlopen(req, timeout=30) as r:
        return json.loads(r.read().decode())

print(post({"action": "register", "label": LABEL, "endpoint": PUBLIC,
            "gpu": GPU, "secret": SECRET}))
print("registered — this machine is now in the order's GPU pool.")

try:
    while True:
        time.sleep(120)
        try: post({"action": "heartbeat", "endpoint": PUBLIC, "secret": SECRET})
        except Exception as e: print("heartbeat failed:", e)
except KeyboardInterrupt:
    post({"action": "retire", "endpoint": PUBLIC, "secret": SECRET})
    print("retired from the pool.")